In [5]:
from datetime import date

CURRENCY_RATES = {
    "EUR": 1.00,
    "USD": 0.92,
    "GBP": 1.17,
    "SEK": 0.086,
    "JPY": 0.0062
}

CURRENCY_SYMBOLS = {
    "EUR": "€",
    "USD": "$",
    "GBP": "£",
    "SEK": "kr",
    "JPY": "¥"
}


def format_currency(amount, currency="EUR"):
    symbol = CURRENCY_SYMBOLS.get(currency, currency)
    return f"{symbol}{amount:.2f}"


def currency_converter(amount, from_currency, to_currency="EUR"):
    amount_in_eur = amount * CURRENCY_RATES[from_currency]

    if to_currency == "EUR":
        return round(amount_in_eur, 2)

    converted_amount = amount_in_eur / CURRENCY_RATES[to_currency]
    return round(converted_amount, 2)


def choose_currency(prompt_text="Choose currency:"):
    currencies = list(CURRENCY_RATES.keys())

    while True:
        print(f"\n{prompt_text}")
        for i, currency in enumerate(currencies, start=1):
            print(f"{i}. {currency}")

        choice = input("Enter currency number: ").strip()

        if not choice.isdigit():
            print("Error! Please enter a valid number.")
            continue

        choice = int(choice)

        if 1 <= choice <= len(currencies):
            return currencies[choice - 1]

        print("Error! Invalid currency number.")


def get_group_name():
    while True:
        group_name = input("Enter group name / location: ").strip()
        if group_name:
            return group_name
        print("Error! Group name cannot be empty.")


def group_size_confirm():
    while True:
        number_of_members = input("Please enter the number of people in your group (1-9): ").strip()

        if not number_of_members.isdigit():
            print("Error! Only numbers 1-9 are allowed!")
            continue

        number_of_members = int(number_of_members)

        if 1 <= number_of_members <= 9:
            return number_of_members

        print("Error! Group size must be between 1 and 9.")


def get_person_name(existing_people):
    while True:
        person = input("Please provide name: ").strip()

        if not person.isalpha():
            print("Error! Only letters a-z allowed!")
            continue

        if len(person) > 10:
            print("Error! Only 10 characters allowed!")
            continue

        if person in existing_people:
            print("Error! This name already exists in the group.")
            continue

        return person


def add_people():
    group_name = get_group_name()
    number_of_members = group_size_confirm()
    display_currency = choose_currency("Choose group display currency:")
    people = []

    print(f"\nEnter {number_of_members} member name(s):")
    while len(people) < number_of_members:
        person = get_person_name(people)
        people.append(person)
        print(f"Added: {person}")

    return group_name, people, display_currency


def get_valid_amount():
    while True:
        raw = input("Enter amount (e.g. 12.50): ").strip()

        try:
            amount = float(raw)
            if amount <= 0:
                print("Error! Amount must be greater than 0.")
                continue
            return round(amount, 2)
        except ValueError:
            print("Error! Enter a valid number, e.g. 12.50")


def choose_category():
    categories = ["food", "transport", "rent", "entertainment", "shopping", "other"]

    while True:
        print("\nChoose a category:")
        for i, category in enumerate(categories, start=1):
            print(f"{i}. {category}")

        choice = input("Enter category number: ").strip()

        if not choice.isdigit():
            print("Error! Please enter a valid number.")
            continue

        choice = int(choice)
        if 1 <= choice <= len(categories):
            return categories[choice - 1]

        print("Error! Invalid category number.")


def get_date():
    today = str(date.today())
    user_input = input(f"Enter date (YYYY-MM-DD) or press Enter for today [{today}]: ").strip()

    if user_input == "":
        return today

    if len(user_input) == 10 and user_input[4] == "-" and user_input[7] == "-":
        year, month, day = user_input.split("-")
        if year.isdigit() and month.isdigit() and day.isdigit():
            return user_input

    print("Invalid format, using today's date.")
    return today


def get_reimbursable():
    while True:
        answer = input("Should this expense be paid back? (yes/no): ").strip().lower()

        if answer in ["yes", "y"]:
            return True
        if answer in ["no", "n"]:
            return False

        print("Error! Please enter yes or no.")


def get_valid_person(people, prompt):
    while True:
        person = input(prompt).strip()
        if person in people:
            return person

        print("Error! Person must be one of the group members:")
        print(", ".join(people))


def choose_shared_with(people):
    while True:
        print("\nWho shared this expense?")
        print("Enter member numbers separated by commas, e.g. 1,2")

        for i, person in enumerate(people, start=1):
            print(f"{i}. {person}")

        selection = input("Shared with: ").strip()

        if selection == "":
            print("Error! Please select at least one person.")
            continue

        parts = [part.strip() for part in selection.split(",")]
        chosen_indexes = []
        valid = True

        for part in parts:
            if not part.isdigit():
                valid = False
                break

            index = int(part)

            if index < 1 or index > len(people):
                valid = False
                break

            if index not in chosen_indexes:
                chosen_indexes.append(index)

        if not valid or not chosen_indexes:
            print("Error! Please enter valid member numbers.")
            continue

        shared_with = [people[index - 1] for index in chosen_indexes]
        return shared_with


def add_expense(people):
    print("\n--- Add Expense ---")

    paid_by = get_valid_person(people, "Who paid? ")
    currency = choose_currency("Choose expense currency:")
    original_amount = get_valid_amount()

    description = input("Enter expense description: ").strip()
    if description == "":
        description = "No description"

    category = choose_category()
    expense_date = get_date()
    reimbursable = get_reimbursable()

    if reimbursable:
        shared_with = choose_shared_with(people)
    else:
        shared_with = []

    amount = currency_converter(original_amount, currency, "EUR")

    expense = {
        "paid_by": paid_by,
        "amount": amount,  # stored internally in EUR
        "original_amount": original_amount,
        "currency": currency,
        "description": description,
        "shared_with": shared_with,
        "category": category,
        "date": expense_date,
        "reimbursable": reimbursable,
        "settled": False
    }

    print("\nExpense added successfully.")
    return expense


def show_expenses(expenses):
    print("\n=== ALL EXPENSES ===")

    if not expenses:
        print("No expenses recorded yet.")
        return

    for i, expense in enumerate(expenses, start=1):
        shared_with = ", ".join(expense["shared_with"]) if expense["shared_with"] else "Not applicable"
        reimbursable_status = "Yes" if expense["reimbursable"] else "No"
        settled_status = "Yes" if expense["settled"] else "No"

        print(f"\nExpense #{i}")
        print(f"Description   : {expense['description']}")
        print(f"Paid by       : {expense['paid_by']}")
        print(f"Amount        : {format_currency(expense['original_amount'], expense['currency'])}")
        print(f"Category      : {expense['category']}")
        print(f"Date          : {expense['date']}")
        print(f"Shared with   : {shared_with}")
        print(f"Reimbursable  : {reimbursable_status}")
        print(f"Settled       : {settled_status}")


def simplify_settlements(balances):
    debtors = []
    creditors = []
    settlements = []

    for person, balance in balances.items():
        balance = round(balance, 2)

        if balance < -0.01:
            debtors.append([person, round(-balance, 2)])
        elif balance > 0.01:
            creditors.append([person, round(balance, 2)])

    debtors.sort(key=lambda x: x[1], reverse=True)
    creditors.sort(key=lambda x: x[1], reverse=True)

    while debtors and creditors:
        debtor_name, debt = debtors[0]
        creditor_name, credit = creditors[0]

        amount = round(min(debt, credit), 2)

        settlements.append({
            "from": debtor_name,
            "to": creditor_name,
            "amount": amount  # stored in EUR
        })

        debtors[0][1] = round(debtors[0][1] - amount, 2)
        creditors[0][1] = round(creditors[0][1] - amount, 2)

        if abs(debtors[0][1]) < 0.01:
            debtors.pop(0)

        if abs(creditors[0][1]) < 0.01:
            creditors.pop(0)

    return settlements


def show_settlements(settlements, display_currency):
    print("\n=== SETTLEMENT PLAN ===")

    if not settlements:
        print("No settlements needed.")
        return

    for settlement in settlements:
        payer = settlement["from"]
        receiver = settlement["to"]
        converted_amount = currency_converter(settlement["amount"], "EUR", display_currency)
        print(f"{payer} pays {receiver} {format_currency(converted_amount, display_currency)}")


def show_balances(balances, display_currency):
    print("\n=== BALANCES ===")

    if not balances:
        print("No balances to show.")
        return

    anyone_has_balance = False

    for person, balance in balances.items():
        if balance > 0.01:
            converted_balance = currency_converter(balance, "EUR", display_currency)
            print(f"{person} should receive {format_currency(converted_balance, display_currency)}")
            anyone_has_balance = True
        elif balance < -0.01:
            converted_balance = currency_converter(abs(balance), "EUR", display_currency)
            print(f"{person} owes {format_currency(converted_balance, display_currency)}")
            anyone_has_balance = True

    if not anyone_has_balance:
        print("All balances are settled.")


def confirm_settlement():
    while True:
        answer = input("\nHas it been settled? (y/n): ").strip().lower()

        if answer in ["y", "yes"]:
            return True
        if answer in ["n", "no"]:
            return False

        print("Error! Please enter y or n.")


def split_amount_fairly(amount, num_people, start_index):
    amount_in_cents = int(round(amount * 100))
    base_share_cents = amount_in_cents // num_people
    remainder_cents = amount_in_cents % num_people

    shares_in_cents = [base_share_cents] * num_people

    for i in range(remainder_cents):
        index = (start_index + i) % num_people
        shares_in_cents[index] += 1

    shares = [share / 100 for share in shares_in_cents]
    return shares


def calculate_balances(people, expenses):
    balances = {person: 0.0 for person in people}
    active_expense_index = 0

    for expense in expenses:
        if not expense["reimbursable"] or expense["settled"]:
            continue

        payer = expense["paid_by"]
        amount = round(expense["amount"], 2)  # EUR
        shared_with = expense["shared_with"]
        num_people = len(shared_with)

        start_index = active_expense_index % num_people
        shares = split_amount_fairly(amount, num_people, start_index)

        for person, share in zip(shared_with, shares):
            balances[person] = round(balances[person] - share, 2)

        balances[payer] = round(balances[payer] + amount, 2)
        active_expense_index += 1

    return balances


def mark_active_expenses_as_settled(expenses):
    any_marked = False

    for expense in expenses:
        if expense["reimbursable"] and not expense["settled"]:
            expense["settled"] = True
            any_marked = True

    return any_marked


def menu():
    print("=== Welcome to Expense Splitter ===\n")

    while True:
        group_name, people, display_currency = add_people()
        expenses = []

        print(f"\nGroup '{group_name}' created successfully.")
        print(f"Members: {', '.join(people)}")
        print(f"Display currency: {display_currency}")

        while True:
            print("\n=== MAIN MENU ===")
            print("1. Add expense")
            print("2. View all expenses")
            print("3. View balances")
            print("4. View settlement plan")
            print("5. Restart with new group")
            print("6. Exit application")

            choice = input("Choose an option (1-6): ").strip()

            if choice == "1":
                expense = add_expense(people)
                expenses.append(expense)
                print("\nExpense saved successfully.")

            elif choice == "2":
                show_expenses(expenses)

            elif choice == "3":
                if not expenses:
                    print("\nNo expenses recorded yet.")
                else:
                    balances = calculate_balances(people, expenses)
                    show_balances(balances, display_currency)

            elif choice == "4":
                if not expenses:
                    print("\nNo expenses recorded yet.")
                else:
                    balances = calculate_balances(people, expenses)
                    settlements = simplify_settlements(balances)
                    show_settlements(settlements, display_currency)

                    if settlements:
                        settled = confirm_settlement()
                        if settled:
                            changed = mark_active_expenses_as_settled(expenses)
                            if changed:
                                print("\nBalances settled. Active expenses marked as settled.")
                            else:
                                print("\nNo active reimbursable expenses were found.")
                        else:
                            print("\nSettlement not confirmed. Records kept.")

            elif choice == "5":
                print("\nRestarting with a new group...\n")
                break

            elif choice == "6":
                print("\nExiting Expense Splitter. Goodbye.")
                return

            else:
                print("Error! Please choose a valid option from 1 to 6.")


def main():
    menu()


if __name__ == "__main__":
    main()

=== Welcome to Expense Splitter ===

Error! Only numbers 1-9 are allowed!
Error! Only numbers 1-9 are allowed!
Error! Only numbers 1-9 are allowed!

Choose group display currency:
1. EUR
2. USD
3. GBP
4. SEK
5. JPY
Error! Please enter a valid number.

Choose group display currency:
1. EUR
2. USD
3. GBP
4. SEK
5. JPY

Enter 3 member name(s):
Added: Ikra
Added: Mike
Added: Rabbil

Group 'Athens' created successfully.
Members: Ikra, Mike, Rabbil
Display currency: EUR

=== MAIN MENU ===
1. Add expense
2. View all expenses
3. View balances
4. View settlement plan
5. Restart with new group
6. Exit application

--- Add Expense ---

Choose expense currency:
1. EUR
2. USD
3. GBP
4. SEK
5. JPY
Error! Invalid currency number.

Choose expense currency:
1. EUR
2. USD
3. GBP
4. SEK
5. JPY

Choose a category:
1. food
2. transport
3. rent
4. entertainment
5. shopping
6. other

Who shared this expense?
Enter member numbers separated by commas, e.g. 1,2
1. Ikra
2. Mike
3. Rabbil

Expense added successfu

In [ ]:
3

3
